# 06 — Implement STAC

Package a classified raster as portable, validated STAC metadata using PySTAC extension APIs.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/06-implement-stac/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/06-implement-stac/lesson.md) · [Exercise](../modules/06-implement-stac/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/06-implement-stac"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Prepare a mask and package it

The CLI consumes a staged source `Directory` and a water-mask `File`. STAC preserves source identity and acquisition time while describing the cropped product grid.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON

waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band green
waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band nir
waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif
waterbodies otsu --raster norm_diff.tif
waterbodies stac --input-item source --water-body otsu.tif
)


## Inspect through PySTAC

Use the projection and raster extension accessors to read metadata. Resolve the asset href against the Item's location; a relative href must lead to a readable mask.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from pystac.extensions.projection import ProjectionExtension
from pystac.extensions.raster import RasterExtension
from waterbodies.common import read_item
from waterbodies.validation import validate_item
source = read_item(Path('source'))
item = read_item(Path('catalog'))
validate_item(item)
asset = item.assets['data']
href = asset.get_absolute_href()
print('Identity/time:', item.id, item.datetime)
print('Footprint:', item.bbox)
print('Projection:', ProjectionExtension.ext(item).shape)
print('Raster bands:', RasterExtension.ext(asset).bands)
print('Relative asset:', asset.href, 'Resolved:', href)
assert item.id == source.id and item.datetime == source.datetime
assert item.bbox == [1, 1, 7, 7]
assert ProjectionExtension.ext(item).shape == [6, 6]
assert href is not None and Path(href).is_file()
print('STAC validation passed')
PYTHON
)


## Deliberately violate an extension constraint

Clone the Item in memory and use PySTAC's projection setter to supply an invalid one-element shape. This leaves the saved product intact. Validation must reject it, including the declared extension schema.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import pystac
from pystac.extensions.projection import ProjectionExtension
from waterbodies.common import read_item
from waterbodies.validation import validate_item
invalid = read_item(Path('catalog')).clone()
ProjectionExtension.ext(invalid).shape = [6]
try:
    validate_item(invalid)
except pystac.errors.STACValidationError as error:
    print('Expected projection validation error:', error)
else:
    raise AssertionError('Invalid projection metadata must fail validation')
PYTHON
python -m pytest "$WORKSPACE/tests/tools/test_processing.py" -k "complete or validation"
)


## Record your findings

**Expected outcome:** The catalog contains a validated Item, preserved identity/time, cropped projection metadata and a resolvable relative mask asset.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/06-implement-stac/solution.md) after the activity. Return to [the lab index](README.md).
